In [0]:
dbutils.library.restartPython()

In [0]:
import pyspark.sql.functions as f
from delta.tables import DeltaTable
from common_utils.logging import get_logger
from common_utils.scd_functions import scd_type1
from common_utils.generic_functions import rename_columns , casting , drop_columns
import json

In [0]:
# =========================================
# 0.Logger
# =========================================
logger = get_logger("b2s_customers-process.ipynb")
logger.info("session_started")

# =========================================
# 1. CONFIG PATH
# =========================================
dbutils.widgets.text("path", "")
config_path =dbutils.widgets.get("path") 
print(config_path)
logger.info("setting the configurations")


# =========================================
# 2. Read Config
# =========================================
with open(config_path, "r") as file:
  config = json.load(file) 

# =========================================
# 3. CONFIG Sections
# =========================================
source_config = config['source']
source_name = config["source"]["source_name"]
target_table = config["target"]["target_table"]



logger.info("Configuration has been done")

from pyspark.sql import Window

# ============================================================
# 4. Customer Transformations
# ============================================================


def customer_transformations(df,config):
 # --------------------------------------------------------
 #Correcting customers name
 # --------------------------------------------------------
 if "customer_name" in df.columns:
     logger.info("Starting with correcting customers name form [%s] ", source_name)
     df= (df.withColumn("customer_name",f.lower("customer_name"))
                   .withColumn("customer_name",f.split(f.col("customer_name"),","))
                   .withColumn("first_name",f.expr("get(customer_name,1)"))
                   .withColumn("last_name",f.expr("get(customer_name,0)"))
                   )
 # --------------------------------------------------------
 #City
 # --------------------------------------------------------


 if "city" in df.columns:
     logger.info("Standering city column for [%s] source", source_name)
     df = df.withColumn("city",f.lower(f.col("city")))



 # --------------------------------------------------------
 # Postcode
 # --------------------------------------------------------
 if "postcode" in df.columns:
     logger.info("Removing points values from postcode column for [%s] source", source_name)
     df = df.withColumn("postcode",f.regexp_replace(f.col("postcode"),r"\.0$",""))

    
 # --------------------------------------------------------
 #chaning valid from and valid to types
 # --------------------------------------------------------
 if "valid_from" in df.columns:
     logger.info("Changing valid_from and valid_to columns types for [%s] source", source_name)
     df = df.withColumn("valid_from",f.from_unixtime(f.expr("try_cast(valid_from as bigint)")))
 if "valid_to" in df.columns:
     logger.info("Changing valid_from and valid_to columns types for [%s] source", source_name)
     df = df.withColumn("valid_to",f.from_unixtime(f.expr("try_cast(valid_to as bigint)")))

 # --------------------------------------------------------
 #Adding new Country column
 # --------------------------------------------------------
 df= df.withColumn("country",f.lit("USA"))



 # --------------------------------------------------------
 # Rename columns
 # --------------------------------------------------------

 df = rename_columns(df,config.get("rename_columns",{}))

 # --------------------------------------------------------
 # Casting columns
 # --------------------------------------------------------
 
 df = casting(df,config.get("cast_columns",{}))

 # --------------------------------------------------------
 # Drop Columns
 # --------------------------------------------------------
 
 drop_columns = [
        c for c in config.get("drop_columns", [])
        if c in df.columns
    ]

 if drop_columns:
     logger.info(
         "[%s] Dropping columns: %s",
         source_name,
         drop_columns
     )
     df = df.drop(*drop_columns)

 # --------------------------------------------------------
 # Deduplication
 # --------------------------------------------------------

 dedup = config.get("dedplication",{})
 if dedup.get("enabled",False):
     logger.info("removing duplicates from the [%s]", source_name)
     order_expression = []
     for item in dedup.get("order_by",[]):
         col = f.col(item["column"])
         direction = item.get("direction","asc").lower()
         nulls = item.get("nulls", "last").lower()

         if direction == "desc":
             col = (
                 col.desc_nulls_last()
                 if nulls =="last"
                 else col.desc_nulls_first()
             )
         else:
             col= (
                 col.asc_nulls_last()
                 if nulls == "last"
                 else col.asc_nulls_first
             )
         order_expression.append(col)
     #simple if/else if null = "last"L
     # col = col.desc_nulls_last()
     #else:
     #      col=col.desc_nulls_first()
     window = Window.partitionBy(*dedup.get("key_columns")).orderBy(*order_expression)
     df = df.withColumn("rn", f.row_number().over(window)).filter(f.col("rn")==1).drop("rn")

 # --------------------------------------------------------
 # Select final columns
 # --------------------------------------------------------

 logger.info(
     "[%s] Selecting final silver columns",
     source_name
 )
 df = df.select(*config.get("select_columns",[]))
 return df


# ============================================================
# 5. Reading Bronze Table
# ============================================================
logger.info("reading from source table [%s]",source_config["source_table"])
df = spark.read.table(source_config["source_table"])

# ============================================================
# 6. Run Transformations
# ============================================================

cleaned_df = customer_transformations(
    df,config
)

logger.info(
    "[%s] Sales transformation completed",
    source_name
)



# ============================================================
# 7. SCD Type 1 / Initial Load
# ============================================================

scd_type = config.get("scd_type",1)
key_columns = config.get("scd").get("key_columns",[])

if spark.catalog.tableExists(target_table):
    if scd_type == 1:
        logger.info(
            "[%s] Starting SCD Type 1",
            source_name
        )
        scd_type1(
            spark,cleaned_df,target_table,key_columns
        )
    else:
        raise ValueError(
            f"Unsupported SCD type [{scd_type}] "
            f"for source [{source_config['source_name']}]"
        )
else:
    spark.sql(
        "CREATE SCHEMA IF NOT EXISTS retail_data_platform.silver"
    )
    (
        cleaned_df
        .write
        .format("delta")
        .mode("overwrite").saveAsTable(target_table)
    )

logger.info(
    "successfully wrote to table [%s]", target_table
)

